In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType


/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *

In [3]:
with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_matrices.pickle',
           'rb') as f:
    pg_dict = pickle.load(f)

In [4]:
# inserts mutation amino acid in corresponding position in protein sequence
def insert_wt(seq, pos, wt_aa):
    seq_list = list(seq)
    pos = int(pos)
    if pos < len(seq_list):
        seq_list[pos] = wt_aa
    return ''.join(seq_list)

# computes the ranking loss between two iterables
def listwise_ranking_loss(preds, targets):
    indices = targets.sort(descending=True).indices
    preds = torch.gather(preds, dim=-1, index=indices)
    cumsums = preds.exp().flip(dims=[-1]).cumsum(dim=-1).flip(dims=[-1])
    loss = torch.log(cumsums + 1e-10) - preds
    return loss.mean()

In [ ]:
# load_dotenv()
# cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

# client = storage.Client()
# bucket = client.bucket('domainome-data')
# blob = bucket.blob('SupplementaryTable2.txt')

# df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

Get data for one domain.  In this case P07316_PF00030_87.  Translate position by initial domain position.

In [ ]:
# df_one_protein = df.where(df['domain_ID'] == 'P07316_PF00030_87').dropna()

# dom_position = df_one_protein['position'] - 87.0
# df_one_protein.insert(loc=0, column='real_position', value=dom_position)
# df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

# df_one_protein_ns.head()



,real_position,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
121785,0.0,P07316_PF00030_87,P07316,AAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,A,False,645.0,476.0,327.0,202.0,183.0,264.0,482.6667,0.067864,0.006483,0.163960,0.070917,385.0
121786,0.0,P07316_PF00030_87,P07316,CAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,C,False,1155.0,733.0,502.0,198.0,257.0,578.0,796.6667,0.061549,0.005723,0.094874,0.062609,385.0
121787,0.0,P07316_PF00030_87,P07316,DAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,D,False,1079.0,784.0,566.0,197.0,220.0,399.0,809.6667,0.054534,0.006000,0.018137,0.065632,385.0
121788,0.0,P07316_PF00030_87,P07316,EAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,E,False,305.0,251.0,187.0,75.0,94.0,182.0,247.6667,0.067474,0.008372,0.159687,0.091586,385.0
121789,0.0,P07316_PF00030_87,P07316,FAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,F,False,803.0,628.0,463.0,241.0,412.0,729.0,631.3333,0.083318,0.004906,0.333017,0.053670,385.0


Insert mutation into correct position of protein sequence.

In [ ]:
# df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
#                                                       insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
#                                                       axis=1)


In [ ]:
# df_mutation = df_one_protein_ns[['real_position','mut_aa','normalized_fitness']]
# df_mutation
# df_mutation.to_csv("mutation.csv")

,real_position,mut_aa,normalized_fitness
121785,0.0,A,0.163960
121786,0.0,C,0.094874
121787,0.0,D,0.018137
121788,0.0,E,0.159687
121789,0.0,F,0.333017
...,...,...,...
123520,0.0,S,0.000670
123521,0.0,T,0.058108
123522,0.0,V,0.049949
123523,0.0,W,0.022983


In [5]:
df_mutation = pd.read_csv('mutation.csv')

Initializing base model Pro Gen 2

In [6]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = 'cpu'
print(f"Using {device} device")

model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


Initializing LoRA

In [7]:
for name, module in base_model.named_modules():
    print(name)


transformer
transformer.wte
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.attn_dropout
transformer.h.0.attn.resid_dropout
transformer.h.0.attn.qkv_proj
transformer.h.0.attn.out_proj
transformer.h.0.mlp
transformer.h.0.mlp.fc_in
transformer.h.0.mlp.fc_out
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.attn_dropout
transformer.h.1.attn.resid_dropout
transformer.h.1.attn.qkv_proj
transformer.h.1.attn.out_proj
transformer.h.1.mlp
transformer.h.1.mlp.fc_in
transformer.h.1.mlp.fc_out
transformer.h.1.mlp.act
transformer.h.1.mlp.dropout
transformer.h.2
transformer.h.2.ln_1
transformer.h.2.attn
transformer.h.2.attn.attn_dropout
transformer.h.2.attn.resid_dropout
transformer.h.2.attn.qkv_proj
transformer.h.2.attn.out_proj
transformer.h.2.mlp
transformer.h.2.mlp.fc_in
transformer.h.2.mlp.fc_out
transformer.h.2.mlp.act
transformer.h.2.mlp.dropout
tran

In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # target_modules=["query", "key", "value", "output.dense"],
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(base_model, lora_config)

In [9]:
for name, module in base_model.named_modules():
    if "qkv_proj" in name or "out_proj" in name:
        print(name, module)

transformer.h.0.attn.qkv_proj lora.Linear(
  (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.1, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=1536, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=4608, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)
transformer.h.0.attn.qkv_proj.base_layer Linear(in_features=1536, out_features=4608, bias=False)
transformer.h.0.attn.qkv_proj.lora_dropout ModuleDict(
  (default): Dropout(p=0.1, inplace=False)
)
transformer.h.0.attn.qkv_proj.lora_dropout.default Dropout(p=0.1, inplace=False)
transformer.h.0.attn.qkv_proj.lora_A ModuleDict(
  (default): Linear(in_features=1536, out_features=8, bias=False)
)
transformer.h.0.attn.qkv_proj.lora_A.default Linear(in_features=1536, out_features=8, bias=False)
tra

Set device and optimizer

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
# optimizer = torch.optim.AdamW(
#     filter(lambda p: p.requires_grad, model.parameters()), lr=1e-2
# )

Load protein sequence and experimental data

In [11]:
protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
fitness_data = df_mutation

In [12]:
len(protein_seq)

90

Making experimental tensor

In [13]:
fitness_data.reset_index(drop=True, inplace=True)

positions = np.arange(len(protein_seq))
# amino_acids = ['L', 'A', 'G', 'V', 'S', 'E', 'R', 'T', 'I',
#                'D', 'P', 'K', 'Q', 'N', 'F', 'Y', 'M', 'H', 'W', 'C']

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

Construct df2

In [14]:
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})

In [15]:
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')

Construct experimental tensor

In [16]:
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

Set up train, validate, test

In [17]:
all_indices = torch.randperm(seq_len)

num_train = int(seq_len * 0.5)
num_val   = int(seq_len * 0.25)
num_test  = seq_len - num_train - num_val

train_indices = all_indices[:num_train] # len 900
val_indices   = all_indices[num_train:num_train + num_val]
test_indices  = all_indices[num_train + num_val:]


Fine-tuning model

In [19]:
train_losses = []
val_losses = []
early_stop_count = 0


vocab_dict = tokenizer.get_vocab()
seq_list = list(protein_seq)
inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

for epoch in range(10):
    model.train()
    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)


    # Compute LLR ----
    # wt_logits = torch.log_softmax(logits, dim=-1)
    # wt_logits = wt_logits[1:-1, :]

    wt_logits = torch.log_softmax(logits, dim=-1)
    # wt_logits = wt_logits[1:-1, :]

    residue_indices = torch.arange(len(protein_seq))


    # ignoring BOS token
    seq_indices = [vocab_dict[aa] for aa in seq_list]

    wt_norm_tensor = wt_logits[residue_indices, seq_indices].unsqueeze(-1)
    LLR_tensor = wt_logits - wt_norm_tensor
    # LLR_tensor_aa_only = LLR_tensor[:, 4:24]
    LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]

    #p sure i can delete this
    #before_flatten = torch.transpose(LLR_tensor_aa_only, 0, 1)

    #flatten the LLR_tensor
    flattened_LLR_tensor = LLR_tensor_aa_only.flatten()
    flattened_exp_tensor = exp_tensor.to(device)
    flattened_LLR_tensor = flattened_LLR_tensor.to(device)

    #to stack
    combined = torch.stack([flattened_LLR_tensor, flattened_exp_tensor], dim=0)

    ft_tensor = torch.transpose(combined, 0, 1)

    predicted_scores = []
    experimental_values = []

    train_tensor = ft_tensor[train_indices]

    #drop nan
    train_tensor = train_tensor[~torch.any(train_tensor.isnan(), dim=1)]

    num_samples = 20
    positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

    #Get columns
    # this may be wrong!!!!
    predicts = positions[:, 1] #LLR
    targets = positions[:, 0] #exp

# Compute loss with predicts and targets
    loss = listwise_ranking_loss(predicts, targets)
    loss.backward()
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(name, param.grad.abs().mean())
    optimizer.step()
    optimizer.zero_grad()

    train_losses.append(loss.item())

    # ----------- VALIDATION (no backprop) -----------
    model.eval()
    with torch.no_grad():
      val_tensor = ft_tensor[val_indices]
      val_tensor = val_tensor [~torch.any(val_tensor.isnan(), dim=1)]

      num_samples = 20
      positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

      predicts = positions[:, 1] #LLR
      targets = positions[:, 0] #exp

      val_loss = listwise_ranking_loss(predicts, targets)
      val_losses.append(val_loss.item())


    print(f"Epoch {epoch+1} - Training Loss: {loss.item():.4f} | Validation Loss: {val_loss.item():.4f}")

    if val_loss.item() > loss.item():
        early_stop_count += 1
    else:
        early_stop_count = 0

    # if early_stop_count > 2:
    #     print("Validation loss exceeded training loss 3 times — early stopping.")
    #     break
    print(early_stop_count)

base_model.model.transformer.h.0.attn.qkv_proj.lora_A.default.weight tensor(0.)
base_model.model.transformer.h.0.attn.qkv_proj.lora_B.default.weight tensor(0.)
base_model.model.transformer.h.0.attn.out_proj.lora_A.default.weight tensor(0.)
base_model.model.transformer.h.0.attn.out_proj.lora_B.default.weight tensor(0.)
base_model.model.transformer.h.1.attn.qkv_proj.lora_A.default.weight tensor(0.)
base_model.model.transformer.h.1.attn.qkv_proj.lora_B.default.weight tensor(0.)
base_model.model.transformer.h.1.attn.out_proj.lora_A.default.weight tensor(0.)
base_model.model.transformer.h.1.attn.out_proj.lora_B.default.weight tensor(0.)
base_model.model.transformer.h.2.attn.qkv_proj.lora_A.default.weight tensor(0.)
base_model.model.transformer.h.2.attn.qkv_proj.lora_B.default.weight tensor(0.)
base_model.model.transformer.h.2.attn.out_proj.lora_A.default.weight tensor(0.)
base_model.model.transformer.h.2.attn.out_proj.lora_B.default.weight tensor(0.)
base_model.model.transformer.h.3.attn.qk

In [20]:
type(model)

peft.peft_model.PeftModelForCausalLM

In [21]:
names = list(pg_dict.keys())

In [22]:
name = 'PAI1_HUMAN'
pg_dict[name]
# pg_dict['S22A1_HUMAN']

{'sequence': 'MQMSPALTCLVLGLALVFGEGSAVHHPPSYVAHLASDFGVRVFQQVAQASKDRNVVFSPYGVASVLAMLQLTTGGETQQQIQAAMGFKIDDKGMAPALRHLYKELMGPWNKDEISTTDAIFVQRDLKLVQGFMPHFFRLFRSTVKQVDFSEVERARFIINDWVKTHTKGMISNLLGKGAVDQLTRLVLVNALYFNGQWKTPFPDSSTHRRLFHKSDGSTVSVPMMAQTNKFNYTEFTTPDGHYYDILELPYHGDTLSMFIAAPYEKEVPLSALTNILSAQLISHWKGNMTRLPRLLVLPKFSLETEVDLRKPLENLGMTDMFRQFQADFTSLSDQEPLHVAQALQKVKIEVNESGTVASSSTAVIVSARMAPEEIIMDRPFLFVVRHNPTGTVLFMGQVMEP',
 'DMS': array([[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.97824751,         nan,         nan, ..., -0.33072981,
         -0.83554036,         nan],
        [-1.0317919 ,         nan,  1.78156975, ..., -0.73628775,
                 nan,  1.07274511],
        [-2.3878408 ,         nan,  0.79816832, ...

In [23]:
sequence = pg_dict[name]['sequence']

In [24]:
model.eval()

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

input = tokenizer(sequence, return_tensors="pt").to(device)
outputs = model(**inputs)

prompt1 = "1"+sequence  # run it forwards
prompt2 = "2"+sequence[::-1]  # run it backwards

input_ids1 = torch.tensor(tokenizer.encode(prompt1)).unsqueeze(0).to(model.device)
with torch.no_grad():
    logits1 = model(input_ids1).logits
shift_logits1 = logits1[:, :-1, :]  # remove last entry

input_ids2 =  torch.tensor(tokenizer.encode(prompt2)).unsqueeze(0).to(model.device)
with torch.no_grad():
    logits2 = model(input_ids2).logits
shift_logits2 = logits2[:, :-1, :] # remove last entry

shift_logits2 = shift_logits2[:, torch.arange(shift_logits2.size(1) - 1, -1, -1), :]

input_ids = input_ids1[:, 1:]

# take averages of matrices, 2nd one in reverse order
# to simulate BERT output

logits = (shift_logits1 + shift_logits2)/2

log_probs = F.log_softmax(logits, dim = -1)
# n = log_probs.size(1)

ref_log_probs = log_probs[0, torch.arange(input_ids.size(1)), input_ids[0]]
ref_log_probs = ref_log_probs.unsqueeze(1)


llr_matrix = log_probs - ref_log_probs
llr_matrix = llr_matrix[0][:, aa_token_ids]
log_probs = log_probs[0][:, aa_token_ids]

In [29]:
model.print_trainable_parameters()

trainable params: 1,990,656 || all params: 766,794,272 || trainable%: 0.2596


In [26]:
type(model)

peft.peft_model.PeftModelForFeatureExtraction

In [25]:
with torch.no_grad():
    out_peft  = model(**inputs).logits
    out_base  = base_model(**inputs).logits

check_tensor = out_peft - out_base

In [26]:
check_tensor

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])

In [27]:
loss.backward()
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.grad.abs().mean())

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.